# Legal Retrieval Pipeline - Minimal Runbook

Run cells from top to bottom. The data-preparation block is grouped in `prepare_training_data`; individual prep stages are not used here. BGE rerank uses top-50 hybrid-router candidates, and optional LLM rerank uses top-20 BGE rerank chunks.


In [ ]:
# Clone code đan run and run by notebook
!rm -rf /kaggle/working/*
!git clone -b v2 --single-branch --depth 1 https://github.com/Dlevinh755/temp
!cp -r /kaggle/working/temp/* /kaggle/working/
!rm -rf /kaggle/working/temp

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print(Path.cwd())

## 1. Install dependencies

Run once per environment.

In [ ]:
!python3 -m pip install -r requirements.txt -q
!python3 -m pip install sentence-transformers faiss-cpu torch -q
# !python3 -m pip install "unsloth[colab-new]" bitsandbytes -q

## 2. Dataset variables

Switch these paths between sample and full data.

In [ ]:
!python3 scripts/download_legalraw.py --output_dir raw_data/legalraw/full

In [ ]:
!rm -rf raw_data/legalraw/sample_100q/*
!python3 scripts/make_legalraw_sample.py --input_dir raw_data/legalraw/full --output_dir raw_data/legalraw/sample_100q --num_questions 100 --num_extra_articles 300 --seed 42

In [ ]:
DATASET_NAME = "legalraw_sample_100q_bge"
CORPUS_PATH = "raw_data/legalraw/sample_100q_bge/legal_corpus.json"
QUESTIONS_PATH = "raw_data/legalraw/sample_100q_bge/train.json"
OUTPUT_DIR = "outputs"

DENSE_MODEL = "BAAI/bge-m3"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
LLM_RERANK_MODEL = "unsloth/Qwen3.5-4B-Base"
LLM_RERANK_BACKEND = "unsloth_causal_lm"
USE_LLM_RERANK = False
DEVICE = "cuda"
BATCH_SIZE = 32
TOP_K = 100
CANDIDATE_TOP_K = 50
LLM_RERANK_TOP_K = 20
BGE_GRAD_ACCUM = 1
RERANK_GRAD_ACCUM = 1
POSITIVE_CHUNKS_PER_AID = 2


## 3. Prepare training data

Runs: `prepare_data -> split -> build_dense_index -> mine_hard_negatives -> sample_negatives`. This first dense index uses the base BGE model for hard-negative mining.

In [ ]:
!python3 run.py  \
  --stage prepare_training_data  \
  --dataset_name {DATASET_NAME} \
  --corpus_path {CORPUS_PATH} \
  --questions_path {QUESTIONS_PATH} \
  --output_dir {OUTPUT_DIR} \
  --dense_model {DENSE_MODEL} \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --top_k {TOP_K} \
  --positive_chunks_per_aid {POSITIVE_CHUNKS_PER_AID}

## 4. Train BGE retriever and rebuild dense index


In [ ]:
!python3 run.py \
  --stage train_bge_retriever \
  --dataset_name {DATASET_NAME} \
  --corpus_path {CORPUS_PATH} \
  --questions_path {QUESTIONS_PATH} \
  --output_dir {OUTPUT_DIR} \
  --dense_model {DENSE_MODEL} \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --top_k {TOP_K} \
  --bge_train_batch_size 4 \
  --bge_grad_accum {BGE_GRAD_ACCUM} \
  --bge_epochs 1 \
  --bge_lr 2e-5 \
  --bge_warmup_ratio 0.1 \
  --bge_max_length 512 \
  --bge_use_amp true \
  --bge_gradient_checkpointing true \
  --bge_auto_batch_reduce true \
  --bge_negatives_per_example 3

## 5. Tune and build BM25

In [ ]:
!python3 run.py \
  --stage tune_and_build_bm25 \
  --dataset_name {DATASET_NAME} \
  --corpus_path {CORPUS_PATH} \
  --questions_path {QUESTIONS_PATH} \
  --output_dir {OUTPUT_DIR} \
  --top_k {TOP_K} \
  --bm25_k1_grid 1.2 \
  --bm25_b_grid 0.9 \
  --bm25_tune_metric recall@10 \
  --use_tuned_bm25 true

## 6. Cache BM25/BGE scores

In [ ]:
!python3 run.py \
  --stage retrieve_cache \
  --dataset_name {DATASET_NAME} \
  --corpus_path {CORPUS_PATH} \
  --questions_path {QUESTIONS_PATH} \
  --output_dir {OUTPUT_DIR} \
  --dense_model {DENSE_MODEL} \
  --device {DEVICE} \
  --batch_size {BATCH_SIZE} \
  --top_k {TOP_K} \
  --use_tuned_bm25 true

## 7. Debug BGE retrieval cache

Run this before router training. It checks whether BGE standalone retrieval has positive hits, whether the dense index/model is correct, and whether evaluation is reading the right cache.

In [ ]:
!python3 scripts/debug_bge_retrieval.py   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --dense_model {DENSE_MODEL}   --device {DEVICE}   --batch_size {BATCH_SIZE}   --top_k {TOP_K}

## 8. Train router and write hybrid caches

Writes fixed alpha `0.5` and predicted-alpha hybrid caches for router/val/test.

In [ ]:
!python3 run.py \
--stage train_router \
--dataset_name {DATASET_NAME} \
--corpus_path {CORPUS_PATH} \
--questions_path {QUESTIONS_PATH} \
--output_dir {OUTPUT_DIR} \
--dense_model {DENSE_MODEL} \
--device {DEVICE} \
--batch_size {BATCH_SIZE} \
--top_k {TOP_K} \
--use_tuned_bm25 true

## 9. Evaluate retrieval baselines

Run once after `train_router` to inspect BM25-only, dense-only, fixed hybrid, and router hybrid before reranking.

In [ ]:
!python3 run.py \
--stage evaluate \
--dataset_name {DATASET_NAME} \
--corpus_path {CORPUS_PATH} \
--questions_path {QUESTIONS_PATH} \
--output_dir {OUTPUT_DIR} \
--dense_model {DENSE_MODEL} \
--device {DEVICE} \
--batch_size {BATCH_SIZE} \
--top_k {TOP_K} \
--use_tuned_bm25 true \
--force true

## 10. Train BGE reranker

Train the stage-2 reranker immediately before applying it to hybrid candidates.

In [ ]:
!python3 run.py   --stage train_reranker   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --rerank_model {RERANK_MODEL}   --device {DEVICE}   --reranker_train_batch_size 4   --reranker_grad_accum {RERANK_GRAD_ACCUM}   --reranker_epochs 1   --reranker_lr 2e-5   --reranker_max_length 512   --reranker_use_amp true

## 11. BGE rerank hybrid candidates

Uses the trained reranker on chunks expanded from the top-50 `hybrid_router` aid candidates, then the next cell evaluates the reranked result at aid level.

In [ ]:
!python3 run.py   --stage rerank_bge   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --rerank_model {OUTPUT_DIR}/{DATASET_NAME}/models/bge_reranker   --device {DEVICE}   --batch_size 16   --candidate_top_k {CANDIDATE_TOP_K}   --force true

## 12. Inspect BGE rerank F2 lift

`rerank_bge` tunes threshold on `val` using `rerank_score`, applies it to `test`, and compares against `hybrid_router`.

In [ ]:
from pathlib import Path
from src.utils.artifact import read_json
root = Path(OUTPUT_DIR) / DATASET_NAME
for split in ['val', 'test']:
    metrics = read_json(root / 'eval' / f'bge_rerank_{split}_metrics.json')
    threshold = metrics['threshold']
    comparison = metrics.get('comparison', {})
    print(split)
    print('  threshold:', threshold['threshold'])
    print('  precision:', threshold['precision'])
    print('  recall:', threshold['recall'])
    print('  f2:', threshold['f2'])
    print('  f2_delta_vs_hybrid_router:', comparison.get('f2_delta'))
    print('  ndcg@10_delta_vs_hybrid_router:', comparison.get('ndcg@10_delta'))
print('threshold summary')
print(read_json(root / 'eval' / 'bge_rerank_threshold.json'))

## 13. Optional: train LLM reranker with Unsloth

Runs only when `USE_LLM_RERANK = True`. This fine-tunes `LLM_RERANK_MODEL` with LoRA using the `unsloth_causal_lm` backend and weak chunk labels from `qwen_train_ready.jsonl`.


In [ ]:
if USE_LLM_RERANK:
    !python3 run.py \
      --stage train_llm_reranker \
      --dataset_name {DATASET_NAME} \
      --corpus_path {CORPUS_PATH} \
      --questions_path {QUESTIONS_PATH} \
      --output_dir {OUTPUT_DIR} \
      --use_llm_rerank true \
      --llm_rerank_model {LLM_RERANK_MODEL} \
      --llm_rerank_backend {LLM_RERANK_BACKEND} \
      --llm_rerank_train_batch_size 16 \
      --llm_rerank_grad_accum 2 \
      --llm_rerank_epochs 1 \
      --llm_rerank_lr 1e-5 \
      --llm_rerank_max_length 555 \
      --llm_rerank_lora_r 16 \
      --llm_rerank_lora_alpha 16 \
      --llm_rerank_batch_size 16 \
      --llm_rerank_load_in_4bit true
else:
    print("[skip] train_llm_reranker: USE_LLM_RERANK is False")


## 14. Optional: LLM rerank BGE top-20 chunks

Reads `bge_rerank_scores_val/test.parquet`, keeps top `LLM_RERANK_TOP_K` chunks per query, and writes `llm_rerank_scores_val/test.parquet`. Score is `sigmoid(logit("1") - logit("0"))`.


In [ ]:
if USE_LLM_RERANK:
    !python3 run.py \
      --stage rerank_llm \
      --dataset_name {DATASET_NAME} \
      --corpus_path {CORPUS_PATH} \
      --questions_path {QUESTIONS_PATH} \
      --output_dir {OUTPUT_DIR} \
      --use_llm_rerank true \
      --llm_rerank_model {LLM_RERANK_MODEL} \
      --llm_rerank_backend {LLM_RERANK_BACKEND} \
      --llm_rerank_top_k {LLM_RERANK_TOP_K} \
      --llm_rerank_max_length 1024 \
      --llm_rerank_load_in_4bit true \
      --force true
else:
    print("[skip] rerank_llm: USE_LLM_RERANK is False")


## 15. Optional: inspect LLM rerank lift


In [ ]:
from pathlib import Path
from src.utils.artifact import read_json
root = Path(OUTPUT_DIR) / DATASET_NAME
if USE_LLM_RERANK and (root / "eval" / "llm_rerank_threshold.json").exists():
    for split in ["val", "test"]:
        metrics = read_json(root / "eval" / f"llm_rerank_{split}_metrics.json")
        threshold = metrics["threshold"]
        topk = metrics.get("topk_tuned", {})
        comparison = metrics.get("comparison", {})
        print(split)
        print("  threshold:", threshold["threshold"])
        print("  f2:", threshold["f2"])
        print("  best_top_k:", topk.get("top_k"))
        print("  topk_f2:", topk.get("f2"))
        print("  f2_delta_vs_bge_rerank:", comparison.get("f2_delta"))
        print("  topk_f2_delta_vs_bge_rerank:", comparison.get("topk_f2_delta"))
        print("  ndcg@10_delta_vs_bge_rerank:", comparison.get("ndcg@10_delta"))
    print("threshold summary")
    print(read_json(root / "eval" / "llm_rerank_threshold.json"))
else:
    print("[skip] no LLM rerank metrics")


## 16. Evaluate after rerank

In [ ]:
!python3 run.py   --stage evaluate   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --dense_model {DENSE_MODEL}   --device {DEVICE}   --batch_size {BATCH_SIZE}   --top_k {TOP_K}   --use_tuned_bm25 true   --force true

## 17. Validate rerank cache and summary

Checks detailed metrics against `summary.json`, verifies val-selected thresholds are reused on test, validates BGE/rerank cache metadata, duplicate chunk candidates, chunk-to-aid mapping, label unit, split leakage, and dense index freshness.

In [ ]:
!python3 scripts/check_summary_consistency.py \
    --dataset_dir {OUTPUT_DIR}/{DATASET_NAME} \
    --questions_path {QUESTIONS_PATH} \
    --top_k {TOP_K} \
    --candidate_top_k {CANDIDATE_TOP_K} \
    --llm_rerank_top_k {LLM_RERANK_TOP_K} \
    --rerank_model {OUTPUT_DIR}/{DATASET_NAME}/models/bge_reranker

## 18. Priority audit for suspicious metrics

Runs the high-priority checks in order: summary thresholds, rerank row counts, duplicate chunks, chunk-to-aid aggregation, rerank sort direction, router alpha convention, score normalization, cache freshness, label unit, and split leakage.


In [ ]:
!python3 scripts/audit_pipeline_priorities.py \
    --dataset_dir {OUTPUT_DIR}/{DATASET_NAME} \
    --questions_path {QUESTIONS_PATH} \
    --top_k {TOP_K} \
    --candidate_top_k {CANDIDATE_TOP_K}

## 19. DEMO UI
